<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/nonpotential_cournot_game_analytical_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis non-potential Cournot game: analytical check

This small notebook isolates the game from the neural DTB algorithm. It has four blocks:

1. choose the number of players;
2. define the price, costs, and payoffs;
3. define the payoff-gradient velocity;
4. derive and verify symmetric-support Nash equilibria.

The implementation follows the vector field used in Yijie's thesis, Appendix B.1. For player $i$, let

$$
R_i(x)=\sum_{j\ne i}x_j
$$

be the total quantity produced by the *other* players. The cost below is the multiplayer extension consistent with the thesis's two-player equations (4.34)--(4.35) and Appendix-B Jacobian:

$$
c_i(x_i,x_{-i})
=d+a x_i-b(1+2\mu)x_iR_i+2b\mu x_iR_i^2.
$$

Then

$$
\Pi_i(x)=\bigl[a-b(x_i+R_i)\bigr]x_i-c_i(x_i,x_{-i}),
\qquad
g_i(x)=\frac{\partial\Pi_i}{\partial x_i}
=-2b x_i+2b\mu R_i(1-R_i).
$$

**Thesis consistency note.** Equation (4.62) prints the full total $x_1+x_2+x_3$ inside the cost. Taken literally, its derivative does not equal the Appendix-B vector field and does not vanish at the equilibria listed on thesis page 58. This notebook uses $R_i$ because it reproduces both the Appendix-B Jacobian and the listed points.

The thesis page 58 also says that symmetry identifies "four equilibria" and then explicitly lists **five** points in 3D. The solver below does not rely on either count; it derives candidates and tests the Nash conditions.

## Block 1 — Dimension and parameters

Change only `DIM` to inspect another number of players. The numerical examples in the thesis use $a=3$, $b=1$, $d=1$, and $\mu=2$.

In [1]:
import itertools
import math
import numpy as np

DIM = 3
A = 3.0
B = 1.0
COST_OFFSET = 1.0
MU = 2.0
TOL = 1e-10

if not isinstance(DIM, int) or DIM < 2:
    raise ValueError("DIM must be an integer greater than or equal to 2.")
if B <= 0 or MU <= 0:
    raise ValueError("B and MU must be positive.")

print({
    "dimension": DIM,
    "a": A,
    "b": B,
    "d": COST_OFFSET,
    "mu": MU,
})

{'dimension': 3, 'a': 3.0, 'b': 1.0, 'd': 1.0, 'mu': 2.0}


## Block 2 — Game setup

The inverse demand is

$$p(x)=a-b\sum_{j=1}^{n}x_j.$$

Each component returned by `payoffs(x)` is one player's payoff $\Pi_i(x)$. The parameter $d$ is an additive cost, so it changes payoff levels but drops out of the velocity and the equilibrium equations.

In [2]:
def _state_array(x):
    x = np.asarray(x, dtype=float)
    if x.shape[-1] != DIM:
        raise ValueError(f"Expected the last dimension to have size {DIM}; got {x.shape}.")
    return x


def inverse_demand(x):
    """p(x) = a - b * sum_j x_j."""
    x = _state_array(x)
    return A - B * x.sum(axis=-1)


def rival_totals(x):
    """R_i(x) = sum_{j != i} x_j for every player i."""
    x = _state_array(x)
    return x.sum(axis=-1, keepdims=True) - x


def player_costs(x):
    """Cost vector consistent with thesis equations (4.34)-(4.35) and Appendix B."""
    x = _state_array(x)
    rivals = rival_totals(x)
    return (
        COST_OFFSET
        + A * x
        - B * (1.0 + 2.0 * MU) * x * rivals
        + 2.0 * B * MU * x * rivals**2
    )


def payoffs(x):
    """Pi_i(x) = p(x) x_i - c_i(x_i, x_-i)."""
    x = _state_array(x)
    return inverse_demand(x)[..., None] * x - player_costs(x)


example_state = np.full(DIM, 0.2)
print("example state:", example_state)
print("price:", inverse_demand(example_state))
print("player costs:", player_costs(example_state))
print("player payoffs:", payoffs(example_state))

example state: [0.2 0.2 0.2]
price: 2.4
player costs: [1.328 1.328 1.328]
player payoffs: [-0.848 -0.848 -0.848]


## Block 3 — Velocity

The raw game velocity is the vector of each player's own-payoff derivative:

$$
g_i(x)=-2b x_i+2b\mu R_i(x)\bigl(1-R_i(x)\bigr).
$$

For the feasible strategy set $x_i\ge 0$, a boundary player cannot move toward a negative quantity. `projected_velocity` therefore removes an outward-pointing negative velocity at $x_i=0$. This distinction matters: a boundary Nash equilibrium can have $g_i<0$ while its feasible velocity is zero.

In [3]:
def raw_velocity(x):
    """Unconstrained payoff-gradient field g_i = partial Pi_i / partial x_i."""
    x = _state_array(x)
    rivals = rival_totals(x)
    return -2.0 * B * x + 2.0 * B * MU * rivals * (1.0 - rivals)


def projected_velocity(x, boundary_tol=TOL):
    """Tangent-cone projection of g onto the feasible orthant x_i >= 0."""
    x = _state_array(x)
    g = raw_velocity(x)
    return np.where(x <= boundary_tol, np.maximum(g, 0.0), g)


print("raw velocity at the example state:", raw_velocity(example_state))
print("projected feasible velocity:", projected_velocity(example_state))

raw velocity at the example state: [0.56 0.56 0.56]
projected feasible velocity: [0.56 0.56 0.56]


## Block 4 — Analytical equilibrium solver

For a support $S$ of $m\ge2$ active players, symmetry sets $x_i=q_m$ on $S$ and $x_i=0$ outside $S$. Active-player stationarity gives

$$
q_m
=\frac{\mu(m-1)-1}{\mu(m-1)^2}.
$$

A candidate is retained only if it satisfies the nonnegative-strategy Nash/KKT conditions

$$
x_i>0\Rightarrow g_i(x)=0,
\qquad
x_i=0\Rightarrow g_i(x)\le0.
$$

This enumerates the symmetry-derived equilibria. It does not claim that no asymmetric equilibria exist for every possible parameter choice.

In [4]:
def symmetric_support_quantity(active_players):
    """Analytical q_m for an equal-quantity support of size m."""
    m = int(active_players)
    if m < 2:
        return None
    denominator = MU * (m - 1) ** 2
    return (MU * (m - 1) - 1.0) / denominator


def nash_kkt_residual(x, tol=TOL):
    """Maximum violation of feasibility, active stationarity, and inactive optimality."""
    x = _state_array(x)
    g = raw_velocity(x)
    active = x > tol
    violations = [float(np.maximum(-x, 0.0).max(initial=0.0))]
    if np.any(active):
        violations.append(float(np.abs(g[active]).max(initial=0.0)))
    if np.any(~active):
        violations.append(float(np.maximum(g[~active], 0.0).max(initial=0.0)))
    return max(violations)


def solve_symmetric_support_equilibria(dim=DIM, tol=TOL):
    """Derive and verify equal-quantity support equilibria in R_+^dim."""
    if dim != DIM:
        raise ValueError("Set DIM in Block 1 and rerun the notebook before solving.")

    equilibria = [{
        "name": "origin",
        "support": (),
        "q": 0.0,
        "point": np.zeros(dim),
    }]

    # Descending m matches the ordering used in the thesis: all-active first.
    for m in range(dim, 1, -1):
        q = symmetric_support_quantity(m)
        if q is None or q <= tol:
            continue
        for support in itertools.combinations(range(dim), m):
            point = np.zeros(dim)
            point[list(support)] = q
            if nash_kkt_residual(point, tol) <= 100.0 * tol:
                equilibria.append({
                    "name": "support_" + "_".join(str(i + 1) for i in support),
                    "support": tuple(i + 1 for i in support),
                    "q": q,
                    "point": point,
                })
    return equilibria


equilibria = solve_symmetric_support_equilibria()
print(f"Derived {len(equilibria)} symmetry-supported Nash equilibria for DIM={DIM}.")
print("Each line shows E_k, support, analytical q_m, point, KKT residual, and feasible-velocity norm.\n")

for k, equilibrium in enumerate(equilibria):
    point = equilibrium["point"]
    kkt = nash_kkt_residual(point)
    velocity_norm = np.linalg.norm(projected_velocity(point))
    point_text = np.array2string(point, precision=10, separator=", ")
    print(
        f"E_{k}: support={equilibrium['support'] or 'empty'}, "
        f"q={equilibrium['q']:.10g}, x={point_text}, "
        f"KKT residual={kkt:.3e}, ||v_feasible||_2={velocity_norm:.3e}"
    )

if DIM == 3:
    print(
        "\nThesis page 58 explicitly lists these five 3D points: the origin, "
        "(3/8,3/8,3/8), and the three permutations of (1/2,1/2,0)."
    )
elif DIM == 5:
    print(
        "\nThe thesis reports seven selected 5D equilibria (support sizes 5 and 4, plus "
        "the origin). This solver also exposes additional symmetry-supported KKT "
        "equilibria; the thesis says its reported list may be incomplete."
    )

Derived 5 symmetry-supported Nash equilibria for DIM=3.
Each line shows E_k, support, analytical q_m, point, KKT residual, and feasible-velocity norm.

E_0: support=empty, q=0, x=[0., 0., 0.], KKT residual=0.000e+00, ||v_feasible||_2=0.000e+00
E_1: support=(1, 2, 3), q=0.375, x=[0.375, 0.375, 0.375], KKT residual=0.000e+00, ||v_feasible||_2=0.000e+00
E_2: support=(1, 2), q=0.5, x=[0.5, 0.5, 0. ], KKT residual=0.000e+00, ||v_feasible||_2=0.000e+00
E_3: support=(1, 3), q=0.5, x=[0.5, 0. , 0.5], KKT residual=0.000e+00, ||v_feasible||_2=0.000e+00
E_4: support=(2, 3), q=0.5, x=[0. , 0.5, 0.5], KKT residual=0.000e+00, ||v_feasible||_2=0.000e+00

Thesis page 58 explicitly lists these five 3D points: the origin, (3/8,3/8,3/8), and the three permutations of (1/2,1/2,0).
